In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from time import perf_counter
from tqdm import tqdm
from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_merton_inputs, prepare_nig_inputs
)
from pd_estim_A.data.cds_df import get_cds_panel
from pd_estim_A.models.nig.nig_apath import build_weekly_calendar_from_panel
from pd_estim_A.models.nig.nig_em_paper import (
    EM_algo,
    update_theta,
    invert_nig_call_price,
    compute_pd_physical,
    compute_pd_risk_neutral,
    nig_call_price,
)

In [2]:
# Paths
print(Path.cwd())
data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# Load data and prepare panels
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file= output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,  # recommended if it works
)

df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

debt_daily = fill_liabilities(bs, df_cal)

ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel consistent with filtered firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()


nig_df, em_cache = prepare_nig_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt, build_em=False)
print(nig_df.head())
print(nig_df.shape)
print(nig_df.describe())

c:\Users\vkeenan\OneDrive - Delft University of Technology\Documents\University\QRM\Accenture Project\code\notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to c:\Users\vkeenan\OneDrive - Delft University of Technology\Documents\University\QRM\Accenture Project\code\notebooks_test\..\data\derived\ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10
    gvkey       date             E          isin  \
0  100022 2012-01-03  3.328431e+10  DE0005190003   
1  100080 2012-01-03  4.268705e+10  DE000BAY0017   
2  100312 2012-01-03  1.469717e+09  DE0007030009   
3  100581 2012-01-03  4.935351e+10  FR0000120321   
4  100957 2012-01-03  2.931

In [3]:
# call cds panel and merge
cds = get_cds_panel(
    project_root= Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)
# ensure types
nig = nig_df.copy()
nig["gvkey"] = nig["gvkey"].astype(str)
nig["date"]  = pd.to_datetime(nig["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"]  = pd.to_datetime(cds["date"])

# keep only firms that exist in BOTH (drop firms with no CDS)
common_gv = sorted(set(nig["gvkey"].unique()) & set(cds["gvkey"].unique()))
nig = nig[nig["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# also drop CDS rows whose dates are outside merged's date range
dmin, dmax = nig["date"].min(), nig["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge-asof onto merged's dates (direction='backward')
nig = nig.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds    = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    nig,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

nig_df = merged_cds.reset_index(drop=True)

print("firms after intersection:", nig_df["gvkey"].nunique())
print("rows after merge:", len(nig_df))
print("date range:", nig_df["date"].min(), "→", nig_df["date"].max())

[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
[get_cds_panel] sheets read: 22 | rows parsed: 67015 | unmapped sheets: 0
firms after intersection: 21
rows after merge: 76508
date range: 2012-01-03 00:00:00 → 2025-12-19 00:00:00


In [ ]:
# =========================
# FULL-RUN BATCH CONFIG
# =========================
EVAL_START_YEAR = 2014
EVAL_END_YEAR = 2024
TRAIN_YEARS = 2
WEEK_ENDING = "W-FRI"

# Which third of the eligible firms to run:
# allowed values: 1, 2, 3
RUN_THIRD = 1

# if True, error if the eligible universe is not exactly 21 firms
STRICT_EXPECT_21_FIRMS = True

START_PARAMS = {
    "alpha": 10.0,
    "beta1": 0.0,
    "delta": 1.0,
    "beta0": 0.0,
}

T_HORIZON = 1.0
EM_MAX_ITER = 10
EM_MIN_ITER = 3
EM_TOL = 1e-3
MIN_TRAIN_ROWS = 250

USE_QUARTERLY_WARM_START = True
RETRY_COLD_IF_WARM_FAILS = True

In [5]:
def prepare_one_firm_panel(df: pd.DataFrame, gvkey: str) -> pd.DataFrame:
    g = (
        df.loc[df["gvkey"].astype(str) == str(gvkey)]
          .copy()
          .sort_values("date")
          .reset_index(drop=True)
    )

    g["gvkey"] = g["gvkey"].astype(str)
    g["date"] = pd.to_datetime(g["date"])

    for c in [c for c in ["E", "L", "r", "cds"] if c in g.columns]:
        g[c] = pd.to_numeric(g[c], errors="coerce")

    g = (
        g.dropna(subset=["date", "E", "L", "r"])
         .query("E > 0 and L > 0")
         .sort_values("date")
         .groupby("date", as_index=False)
         .last()
         .reset_index(drop=True)
    )
    return g

In [6]:
def make_quarterly_nig_windows(
    g_onefirm: pd.DataFrame,
    *,
    eval_start_year: int,
    eval_end_year: int,
    train_years: int = 2,
    week_ending: str = "W-FRI",
) -> list[dict]:
    g = g_onefirm.sort_values("date").copy()
    weekly_all = pd.DatetimeIndex(
        build_weekly_calendar_from_panel(g, week_ending=week_ending)
    )

    eval_start = pd.Timestamp(f"{eval_start_year}-01-01")
    eval_end = pd.Timestamp(f"{eval_end_year}-12-31")

    weekly_eval = weekly_all[(weekly_all >= eval_start) & (weekly_all <= eval_end)]
    if len(weekly_eval) == 0:
        raise ValueError(
            f"No weekly dates found in evaluation range {eval_start_year}-{eval_end_year}."
        )

    anchors_ser = (
        pd.Series(weekly_eval, index=weekly_eval)
          .groupby(weekly_eval.to_period("Q"))
          .min()
    )

    anchors = pd.DatetimeIndex(anchors_ser.values)
    windows = []

    for i, anchor in enumerate(anchors):
        train_start = anchor - pd.DateOffset(years=train_years) + pd.Timedelta(days=1)
        train_end = anchor

        if i < len(anchors) - 1:
            next_anchor = anchors[i + 1]
            score_dates = weekly_eval[(weekly_eval >= anchor) & (weekly_eval < next_anchor)]
        else:
            next_anchor = pd.NaT
            score_dates = weekly_eval[(weekly_eval >= anchor) & (weekly_eval <= eval_end)]

        windows.append({
            "quarter_no": i + 1,
            "anchor_date": pd.Timestamp(anchor),
            "next_anchor": pd.Timestamp(next_anchor) if pd.notna(next_anchor) else pd.NaT,
            "train_start": pd.Timestamp(train_start),
            "train_end": pd.Timestamp(train_end),
            "score_dates": pd.DatetimeIndex(score_dates),
        })

    return windows

In [7]:
def _nig_params_basic_ok(params: dict) -> bool:
    try:
        alpha = float(params["alpha"])
        beta1 = float(params["beta1"])
        delta = float(params["delta"])
        beta0 = float(params["beta0"])
    except Exception:
        return False

    if not np.isfinite(alpha) or not np.isfinite(beta1) or not np.isfinite(delta) or not np.isfinite(beta0):
        return False
    if alpha <= 0.5:
        return False
    if delta <= 0.0:
        return False
    if abs(beta1) >= alpha:
        return False
    return True

In [9]:
def run_one_firm_nig_roll(
    g_onefirm: pd.DataFrame,
    *,
    windows: list[dict],
    start_params: dict,
    T_horizon: float = 1.0,
    em_max_iter: int = 10,
    em_min_iter: int = 3,
    em_tol: float = 1e-3,
    min_train_rows: int = 250,
    show_progress: bool = True,
    use_quarterly_warm_start: bool = True,
    retry_cold_if_warm_fails: bool = True,
):
    g = g_onefirm.copy().sort_values("date").reset_index(drop=True)
    g["date"] = pd.to_datetime(g["date"])
    g_idx = g.set_index("date").sort_index()

    firm_meta = {}
    for c in ["gvkey", "company", "isin", "country_iso"]:
        if c in g.columns and g[c].notna().any():
            firm_meta[c] = g[c].dropna().iloc[0]
        else:
            firm_meta[c] = np.nan

    quarter_rows = []
    weekly_rows = []

    total_t0 = perf_counter()

    windows_iter = tqdm(
        windows,
        desc=f"Firm {firm_meta['gvkey']}",
        unit="quarter",
        leave=False,
    ) if show_progress else windows

    base_start_params = {
        "alpha": float(start_params["alpha"]),
        "beta1": float(start_params["beta1"]),
        "delta": float(start_params["delta"]),
        "beta0": float(start_params["beta0"]),
    }

    current_start_params = dict(base_start_params)

    for w in windows_iter:
        q_no = int(w["quarter_no"])
        anchor = pd.Timestamp(w["anchor_date"])
        print(f"  -> Quarter {q_no} start | anchor={anchor.date()}")
        quarter_t0 = perf_counter()

        q_no = int(w["quarter_no"])
        anchor = pd.Timestamp(w["anchor_date"])
        train_start = pd.Timestamp(w["train_start"])
        train_end = pd.Timestamp(w["train_end"])
        score_dates = pd.DatetimeIndex(w["score_dates"])

        train_df = g[(g["date"] >= train_start) & (g["date"] <= train_end)].copy()

        base_row = {
            "gvkey": firm_meta["gvkey"],
            "company": firm_meta["company"],
            "isin": firm_meta["isin"],
            "country_iso": firm_meta["country_iso"],
            "quarter_no": q_no,
            "anchor_date": anchor,
            "train_start": train_start,
            "train_end": train_end,
            "n_train_rows": int(len(train_df)),
        }

        if len(train_df) < min_train_rows:
            quarter_rows.append({
                **base_row,
                "ok": False,
                "msg": f"Too few daily rows in training window: {len(train_df)}",
                "em_converged": False,
                "em_n_iter": np.nan,
                "alpha": np.nan,
                "beta1": np.nan,
                "delta": np.nan,
                "beta0": np.nan,
                "theta_anchor": np.nan,
                "A_anchor": np.nan,
                "L_anchor": np.nan,
                "r_anchor": np.nan,
                "PD_P_anchor": np.nan,
                "PD_Q_anchor": np.nan,
                "quarter_runtime_sec": perf_counter() - quarter_t0,
                "start_source": "none",
                "warm_start_used": False,
                "warm_start_retry_to_cold": False,
            })
            continue

        # ---------------------------
        # quarter-1 is always cold
        # ---------------------------
        if q_no == 1:
            em_start_params = dict(base_start_params)
            start_source = "cold_base"
            warm_start_used = False
        elif use_quarterly_warm_start and _nig_params_basic_ok(current_start_params):
            em_start_params = dict(current_start_params)
            start_source = "warm_prev_quarter"
            warm_start_used = True
        else:
            em_start_params = dict(base_start_params)
            start_source = "cold_base"
            warm_start_used = False

        warm_retry_to_cold = False

        try:
            try:
                em_out = EM_algo(
                    E_series=train_df["E"].to_numpy(dtype=float),
                    L_face_series=train_df["L"].to_numpy(dtype=float),
                    rf_series=train_df["r"].to_numpy(dtype=float),
                    dates=train_df["date"].to_numpy(),
                    start_params=em_start_params,
                    start_date=None,
                    end_date=None,
                    max_iter=em_max_iter,
                    min_iter=em_min_iter,
                    tol=em_tol,
                )
            except Exception as e_first:
                if warm_start_used and retry_cold_if_warm_fails:
                    warm_retry_to_cold = True
                    em_out = EM_algo(
                        E_series=train_df["E"].to_numpy(dtype=float),
                        L_face_series=train_df["L"].to_numpy(dtype=float),
                        rf_series=train_df["r"].to_numpy(dtype=float),
                        dates=train_df["date"].to_numpy(),
                        start_params=base_start_params,
                        start_date=None,
                        end_date=None,
                        max_iter=em_max_iter,
                        min_iter=em_min_iter,
                        tol=em_tol,
                    )
                    start_source = "warm_failed_then_cold"
                else:
                    raise e_first

            params = dict(em_out["params"])
            A_anchor = float(em_out["A_win"][-1])
            theta_anchor = float(em_out["theta_win"][-1])

            row_anchor = g_idx.loc[anchor]
            L_anchor = float(row_anchor["L"])
            r_anchor = float(row_anchor["r"])
            E_anchor = float(row_anchor["E"])
            cds_anchor = (
                float(row_anchor["cds"])
                if "cds" in row_anchor.index and pd.notna(row_anchor["cds"])
                else np.nan
            )

            params_anchor = dict(params)
            params_anchor["theta"] = theta_anchor

            PD_P_anchor = float(
                compute_pd_physical(A0=A_anchor, L=L_anchor, T=T_horizon, params=params)
            )
            PD_Q_anchor = float(
                compute_pd_risk_neutral(A0=A_anchor, L=L_anchor, T=T_horizon, params=params_anchor)
            )

            quarter_rows.append({
                **base_row,
                "ok": True,
                "msg": "ok",
                "em_converged": bool(em_out["converged"]),
                "em_n_iter": int(em_out["n_iter"]),
                "alpha": float(params["alpha"]),
                "beta1": float(params["beta1"]),
                "delta": float(params["delta"]),
                "beta0": float(params["beta0"]),
                "theta_anchor": theta_anchor,
                "A_anchor": A_anchor,
                "L_anchor": L_anchor,
                "r_anchor": r_anchor,
                "E_anchor": E_anchor,
                "cds_anchor": cds_anchor,
                "PD_P_anchor": PD_P_anchor,
                "PD_Q_anchor": PD_Q_anchor,
                "quarter_runtime_sec": np.nan,
                "start_source": start_source,
                "warm_start_used": bool(warm_start_used),
                "warm_start_retry_to_cold": bool(warm_retry_to_cold),
            })

            if use_quarterly_warm_start and _nig_params_basic_ok(params):
                current_start_params = {
                    "alpha": float(params["alpha"]),
                    "beta1": float(params["beta1"]),
                    "delta": float(params["delta"]),
                    "beta0": float(params["beta0"]),
                }
            else:
                current_start_params = dict(base_start_params)

            prev_score_A = None

            for d in score_dates:
                row_t = g_idx.loc[d]
                E_t = float(row_t["E"])
                L_t = float(row_t["L"])
                r_t = float(row_t["r"])
                cds_t = (
                    float(row_t["cds"])
                    if "cds" in row_t.index and pd.notna(row_t["cds"])
                    else np.nan
                )

                if pd.Timestamp(d) == anchor:
                    A_t = A_anchor
                    theta_t = theta_anchor
                    source = "anchor_em"
                else:
                    theta_t = float(update_theta(params, r_t))
                    params_t_for_inversion = dict(params)
                    params_t_for_inversion["theta"] = theta_t

                    A_t = float(
                        invert_nig_call_price(
                            E_obs=E_t,
                            L_face=L_t,
                            r=r_t,
                            T=T_horizon,
                            params=params_t_for_inversion,
                            discounting="continuous",
                        )
                    )
                    source = "weekly_rescore"

                dlogA = np.nan
                if prev_score_A is not None and np.isfinite(prev_score_A) and prev_score_A > 0 and np.isfinite(A_t) and A_t > 0:
                    dlogA = float(np.log(A_t / prev_score_A))
                prev_score_A = A_t

                params_t = dict(params)
                params_t["theta"] = theta_t

                PD_P_t = float(
                    compute_pd_physical(A0=A_t, L=L_t, T=T_horizon, params=params)
                )
                PD_Q_t = float(
                    compute_pd_risk_neutral(A0=A_t, L=L_t, T=T_horizon, params=params_t)
                )

                weekly_rows.append({
                    "gvkey": firm_meta["gvkey"],
                    "company": firm_meta["company"],
                    "isin": firm_meta["isin"],
                    "country_iso": firm_meta["country_iso"],
                    "quarter_no": q_no,
                    "anchor_date": anchor,
                    "date": pd.Timestamp(d),
                    "source": source,
                    "train_start": train_start,
                    "train_end": train_end,
                    "alpha": float(params["alpha"]),
                    "beta1": float(params["beta1"]),
                    "delta": float(params["delta"]),
                    "beta0": float(params["beta0"]),
                    "theta": float(theta_t),
                    "A_hat": float(A_t),
                    "dlogA": dlogA,
                    "E": E_t,
                    "L": L_t,
                    "r": r_t,
                    "cds": cds_t,
                    "PD_P_1y": PD_P_t,
                    "PD_Q_1y": PD_Q_t,
                    "is_anchor_date": pd.Timestamp(d) == anchor,
                    "refit_id": f"{firm_meta['gvkey']}_{anchor:%Y-%m-%d}",
                    "start_source": start_source,
                    "warm_start_used": bool(warm_start_used),
                    "warm_start_retry_to_cold": bool(warm_retry_to_cold),
                })

            quarter_elapsed = perf_counter() - quarter_t0
            quarter_rows[-1]["quarter_runtime_sec"] = quarter_elapsed

        except Exception as e:
            quarter_rows.append({
                **base_row,
                "ok": False,
                "msg": str(e),
                "em_converged": False,
                "em_n_iter": np.nan,
                "alpha": np.nan,
                "beta1": np.nan,
                "delta": np.nan,
                "beta0": np.nan,
                "theta_anchor": np.nan,
                "A_anchor": np.nan,
                "L_anchor": np.nan,
                "r_anchor": np.nan,
                "PD_P_anchor": np.nan,
                "PD_Q_anchor": np.nan,
                "quarter_runtime_sec": perf_counter() - quarter_t0,
                "start_source": start_source,
                "warm_start_used": bool(warm_start_used),
                "warm_start_retry_to_cold": bool(warm_retry_to_cold),
            })
            current_start_params = dict(base_start_params)
            continue

    total_elapsed = perf_counter() - total_t0

    quarter_df = pd.DataFrame(quarter_rows).sort_values(["anchor_date"]).reset_index(drop=True)
    weekly_df = pd.DataFrame(weekly_rows).sort_values(["date"]).reset_index(drop=True)

    return quarter_df, weekly_df, total_elapsed

In [10]:
def run_nig_mini_batch(
    df_all: pd.DataFrame,
    *,
    gvkeys: list[str],
    eval_start_year: int,
    eval_end_year: int,
    train_years: int,
    week_ending: str,
    start_params: dict,
    T_horizon: float,
    em_max_iter: int,
    em_min_iter: int,
    em_tol: float,
    min_train_rows: int,
    use_quarterly_warm_start: bool,
    retry_cold_if_warm_fails: bool,
):
    all_quarter = []
    all_weekly = []
    batch_meta = []

    t0 = perf_counter()
    n_firms = len(gvkeys)

    for i, gv in enumerate(gvkeys, start=1):
        firm_t0 = perf_counter()
        print(f"\n[{i}/{n_firms}] Starting firm gvkey={gv}")

        try:
            g_one = prepare_one_firm_panel(df_all, gv)

            windows = make_quarterly_nig_windows(
                g_one,
                eval_start_year=eval_start_year,
                eval_end_year=eval_end_year,
                train_years=train_years,
                week_ending=week_ending,
            )

            print(
                f"[{i}/{n_firms}] gvkey={gv} | "
                f"quarter windows={len(windows)} | "
                f"date range={g_one['date'].min().date()} -> {g_one['date'].max().date()}"
            )

            q_df, w_df, elapsed = run_one_firm_nig_roll(
                g_one,
                windows=windows,
                start_params=start_params,
                T_horizon=T_horizon,
                em_max_iter=em_max_iter,
                em_min_iter=em_min_iter,
                em_tol=em_tol,
                min_train_rows=min_train_rows,
                show_progress=True,
                use_quarterly_warm_start=use_quarterly_warm_start,
                retry_cold_if_warm_fails=retry_cold_if_warm_fails,
            )

            all_quarter.append(q_df)
            all_weekly.append(w_df)

            batch_meta.append({
                "gvkey": gv,
                "ok_quarters": int(q_df["ok"].sum()) if "ok" in q_df.columns else 0,
                "n_quarters": len(q_df),
                "n_weekly_rows": len(w_df),
                "runtime_sec": elapsed,
                "company": q_df["company"].dropna().iloc[0] if len(q_df) and q_df["company"].notna().any() else np.nan,
            })

            print(
                f"[{i}/{n_firms}] Finished firm gvkey={gv} | "
                f"ok_quarters={int(q_df['ok'].sum())}/{len(q_df)} | "
                f"weekly_rows={len(w_df)} | "
                f"runtime={elapsed:.2f}s"
            )

        except Exception as e:
            batch_meta.append({
                "gvkey": gv,
                "ok_quarters": 0,
                "n_quarters": 0,
                "n_weekly_rows": 0,
                "runtime_sec": np.nan,
                "company": np.nan,
                "error": str(e),
            })

            print(f"[{i}/{n_firms}] FAILED firm gvkey={gv} | error={e}")

    total_elapsed = perf_counter() - t0

    quarter_panel = pd.concat(all_quarter, ignore_index=True) if all_quarter else pd.DataFrame()
    weekly_panel = pd.concat(all_weekly, ignore_index=True) if all_weekly else pd.DataFrame()
    batch_meta_df = pd.DataFrame(batch_meta)

    print(f"\nMini-batch finished | firms={n_firms} | total_runtime={total_elapsed:.2f}s")

    return quarter_panel, weekly_panel, batch_meta_df, total_elapsed

In [11]:
def _sort_gvkeys_stably(gvkeys):
    """
    Stable deterministic sort:
    - numeric gvkeys sorted numerically
    - non-numeric gvkeys sorted lexicographically after numeric ones
    """
    def sort_key(x):
        s = str(x)
        return (0, int(s)) if s.isdigit() else (1, s)
    return sorted([str(x) for x in gvkeys], key=sort_key)


def find_all_eligible_gvkeys(
    df_all: pd.DataFrame,
    *,
    eval_start_year: int,
    eval_end_year: int,
    train_years: int,
    week_ending: str,
) -> list[str]:
    eligible = []
    all_gvkeys = _sort_gvkeys_stably(df_all["gvkey"].astype(str).dropna().unique())

    for gv in all_gvkeys:
        try:
            g_one = prepare_one_firm_panel(df_all, gv)

            windows = make_quarterly_nig_windows(
                g_one,
                eval_start_year=eval_start_year,
                eval_end_year=eval_end_year,
                train_years=train_years,
                week_ending=week_ending,
            )

            if len(windows) == 0:
                continue

            first_train_start = windows[0]["train_start"]
            if first_train_start >= g_one["date"].min():
                eligible.append(gv)

        except Exception:
            continue

    return _sort_gvkeys_stably(eligible)


def split_gvkeys_into_thirds(gvkeys: list[str]) -> dict[int, list[str]]:
    """
    Split sorted eligible gvkeys into 3 deterministic groups.
    For 21 firms this will give 7 / 7 / 7.
    """
    gvkeys_sorted = _sort_gvkeys_stably(gvkeys)
    chunks = np.array_split(np.array(gvkeys_sorted, dtype=object), 3)
    return {
        1: [str(x) for x in chunks[0].tolist()],
        2: [str(x) for x in chunks[1].tolist()],
        3: [str(x) for x in chunks[2].tolist()],
    }


eligible_gvkeys = find_all_eligible_gvkeys(
    nig_df,
    eval_start_year=EVAL_START_YEAR,
    eval_end_year=EVAL_END_YEAR,
    train_years=TRAIN_YEARS,
    week_ending=WEEK_ENDING,
)

if STRICT_EXPECT_21_FIRMS and len(eligible_gvkeys) != 21:
    raise ValueError(
        f"Expected exactly 21 eligible firms, found {len(eligible_gvkeys)}. "
        f"Set STRICT_EXPECT_21_FIRMS=False if this is intentional."
    )

gvkey_thirds = split_gvkeys_into_thirds(eligible_gvkeys)

if RUN_THIRD not in (1, 2, 3):
    raise ValueError("RUN_THIRD must be one of {1, 2, 3}.")

run_gvkeys = gvkey_thirds[RUN_THIRD]

print(f"Eligible firms found: {len(eligible_gvkeys)}")
print("All eligible gvkeys:")
print(eligible_gvkeys)

print("\nThirds:")
for k in [1, 2, 3]:
    print(f"Third {k} ({len(gvkey_thirds[k])} firms): {gvkey_thirds[k]}")

print(f"\nRunning third {RUN_THIRD}: {run_gvkeys}")

Eligible firms found: 21
All eligible gvkeys:
['14447', '17436', '17452', '19349', '23667', '23671', '61616', '100022', '100080', '100957', '101202', '101204', '101336', '101361', '102296', '201794', '220940', '221244', '221616', '222379', '241456']

Thirds:
Third 1 (7 firms): ['14447', '17436', '17452', '19349', '23667', '23671', '61616']
Third 2 (7 firms): ['100022', '100080', '100957', '101202', '101204', '101336', '101361']
Third 3 (7 firms): ['102296', '201794', '220940', '221244', '221616', '222379', '241456']

Running third 1: ['14447', '17436', '17452', '19349', '23667', '23671', '61616']


In [12]:
def run_nig_mini_batch(
    df_all: pd.DataFrame,
    *,
    gvkeys: list[str],
    eval_start_year: int,
    eval_end_year: int,
    train_years: int,
    week_ending: str,
    start_params: dict,
    T_horizon: float,
    em_max_iter: int,
    em_min_iter: int,
    em_tol: float,
    min_train_rows: int,
    use_quarterly_warm_start: bool,
    retry_cold_if_warm_fails: bool,
):
    all_quarter = []
    all_weekly = []
    batch_meta = []

    t0 = perf_counter()
    n_firms = len(gvkeys)

    for i, gv in enumerate(gvkeys, start=1):
        firm_t0 = perf_counter()
        print(f"\n[{i}/{n_firms}] Starting firm gvkey={gv}")

        try:
            g_one = prepare_one_firm_panel(df_all, gv)

            windows = make_quarterly_nig_windows(
                g_one,
                eval_start_year=eval_start_year,
                eval_end_year=eval_end_year,
                train_years=train_years,
                week_ending=week_ending,
            )

            print(
                f"[{i}/{n_firms}] gvkey={gv} | "
                f"quarter windows={len(windows)} | "
                f"date range={g_one['date'].min().date()} -> {g_one['date'].max().date()}"
            )

            q_df, w_df, elapsed = run_one_firm_nig_roll(
                g_one,
                windows=windows,
                start_params=start_params,
                T_horizon=T_horizon,
                em_max_iter=em_max_iter,
                em_min_iter=em_min_iter,
                em_tol=em_tol,
                min_train_rows=min_train_rows,
                show_progress=True,
                use_quarterly_warm_start=use_quarterly_warm_start,
                retry_cold_if_warm_fails=retry_cold_if_warm_fails,
            )

            all_quarter.append(q_df)
            all_weekly.append(w_df)

            batch_meta.append({
                "gvkey": gv,
                "ok_quarters": int(q_df["ok"].sum()) if "ok" in q_df.columns else 0,
                "n_quarters": len(q_df),
                "n_weekly_rows": len(w_df),
                "runtime_sec": elapsed,
                "company": q_df["company"].dropna().iloc[0] if len(q_df) and q_df["company"].notna().any() else np.nan,
            })

            print(
                f"[{i}/{n_firms}] Finished firm gvkey={gv} | "
                f"ok_quarters={int(q_df['ok'].sum())}/{len(q_df)} | "
                f"weekly_rows={len(w_df)} | "
                f"runtime={elapsed:.2f}s"
            )

        except Exception as e:
            batch_meta.append({
                "gvkey": gv,
                "ok_quarters": 0,
                "n_quarters": 0,
                "n_weekly_rows": 0,
                "runtime_sec": np.nan,
                "company": np.nan,
                "error": str(e),
            })

            print(f"[{i}/{n_firms}] FAILED firm gvkey={gv} | error={e}")

    total_elapsed = perf_counter() - t0

    quarter_panel = pd.concat(all_quarter, ignore_index=True) if all_quarter else pd.DataFrame()
    weekly_panel = pd.concat(all_weekly, ignore_index=True) if all_weekly else pd.DataFrame()
    batch_meta_df = pd.DataFrame(batch_meta)

    print(f"\nMini-batch finished | firms={n_firms} | total_runtime={total_elapsed:.2f}s")

    return quarter_panel, weekly_panel, batch_meta_df, total_elapsed

In [ ]:
batch_quarter_df, batch_weekly_df, batch_meta_df, batch_total_elapsed = run_nig_mini_batch(
    nig_df,
    gvkeys=run_gvkeys,
    eval_start_year=EVAL_START_YEAR,
    eval_end_year=EVAL_END_YEAR,
    train_years=TRAIN_YEARS,
    week_ending=WEEK_ENDING,
    start_params=START_PARAMS,
    T_horizon=T_HORIZON,
    em_max_iter=EM_MAX_ITER,
    em_min_iter=EM_MIN_ITER,
    em_tol=EM_TOL,
    min_train_rows=MIN_TRAIN_ROWS,
    use_quarterly_warm_start=USE_QUARTERLY_WARM_START,
    retry_cold_if_warm_fails=RETRY_COLD_IF_WARM_FAILS,
)

print(f"Batch total runtime: {batch_total_elapsed:.2f} sec")
print("batch_quarter_df shape:", batch_quarter_df.shape)
print("batch_weekly_df shape:", batch_weekly_df.shape)

display(batch_meta_df)
display(batch_quarter_df.head(20))
display(batch_weekly_df.head(20))


[1/7] Starting firm gvkey=14447
[1/7] gvkey=14447 | quarter windows=44 | date range=2012-01-03 -> 2025-12-19


Firm 14447:   0%|          | 0/44 [00:00<?, ?quarter/s]

  -> Quarter 1 start | anchor=2014-01-03


In [ ]:
batch_checks = {
    "n_firms_in_run": batch_weekly_df["gvkey"].nunique(),
    "n_unique_selected_gvkeys": len(run_gvkeys),
    "duplicate_gvkey_date_rows": int(batch_weekly_df.duplicated(["gvkey", "date"]).sum()),
    "duplicate_gvkey_quarter_date_rows": int(batch_weekly_df.duplicated(["gvkey", "quarter_no", "date"]).sum()),
    "all_first_quarters_cold": bool(
        (
            batch_quarter_df.sort_values(["gvkey", "anchor_date"])
                            .groupby("gvkey")
                            .head(1)["start_source"] == "cold_base"
        ).all()
    ),
    "warm_starts_only_after_q1": bool(
        (
            batch_quarter_df.loc[batch_quarter_df["quarter_no"] > 1, "start_source"]
            .isin(["cold_base", "warm_prev_quarter", "warm_failed_then_cold"])
        ).all()
    ),
    "ok_quarter_share": float(batch_quarter_df["ok"].mean()) if len(batch_quarter_df) else np.nan,
    "median_runtime_per_quarter_sec": float(batch_quarter_df["quarter_runtime_sec"].median()) if len(batch_quarter_df) else np.nan,
}

display(pd.Series(batch_checks).to_frame("value"))

display(
    batch_quarter_df.groupby("gvkey", as_index=False).agg(
        n_quarters=("quarter_no", "size"),
        n_ok=("ok", "sum"),
        mean_em_iter=("em_n_iter", "mean"),
        mean_runtime_sec=("quarter_runtime_sec", "mean"),
    )
)

In [ ]:
outdir = Path.cwd() / ".." / "data" / "derived" / "nig_batch_outputs"
outdir.mkdir(parents=True, exist_ok=True)

suffix = f"third{RUN_THIRD}_{EVAL_START_YEAR}_{EVAL_END_YEAR}"

batch_quarter_df.to_csv(outdir / f"nig_quarter_{suffix}.csv", index=False)
batch_weekly_df.to_csv(outdir / f"nig_weekly_{suffix}.csv", index=False)
batch_meta_df.to_csv(outdir / f"nig_meta_{suffix}.csv", index=False)

pd.DataFrame({
    "run_third": [RUN_THIRD],
    "n_selected_firms": [len(run_gvkeys)],
    "selected_gvkeys": [",".join(run_gvkeys)],
    "n_total_eligible_firms": [len(eligible_gvkeys)],
    "eval_start_year": [EVAL_START_YEAR],
    "eval_end_year": [EVAL_END_YEAR],
    "train_years": [TRAIN_YEARS],
}).to_csv(outdir / f"nig_run_manifest_{suffix}.csv", index=False)

print("Saved to:", outdir.resolve())